# Homework 3: Logistic Regression for Predicting Lead Conversion

* Dataset: https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/course_lead_scoring_2026.csv


## Data Preparation

In [38]:
import pandas as pd
import numpy as np

In [39]:
data = 'https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/course_lead_scoring_2026.csv'

In [40]:
!wget $data -O data-week-3.csv 

--2026-10-12 01:24:46--  https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/course_lead_scoring_2026.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.109.133, 185.199.108.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.109.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 278746 (272K) [text/plain]
Saving to: ‘data-week-3.csv’

data-week-3.csv     100%[===================>] 272.21K  --.-KB/s    in 0.06s   

2026-10-12 01:24:46 (4.60 MB/s) - ‘data-week-3.csv’ saved [278746/278746]



In [41]:
df = pd.read_csv('data-week-3.csv')
df.head()

,lead_source,industry,employment_status,location,annual_income,number_of_courses_viewed,interaction_count,lead_score,converted
0,organic_search,technology,employed,europe,88160.0,3,4,0.64,1
1,social_media,technology,employed,south_america,72688.0,3,7,0.72,1
2,referral,retail,employed,europe,44697.0,3,4,0.58,1
3,paid_ads,manufacturing,student,NaN,NaN,3,6,0.57,0
4,referral,manufacturing,student,north_america,24062.0,4,5,0.62,1


In [42]:
df.info

<bound method DataFrame.info of          lead_source       industry employment_status       location  \
0     organic_search     technology          employed         europe   
1       social_media     technology          employed  south_america   
2           referral         retail          employed         europe   
3           paid_ads  manufacturing           student            NaN   
4           referral  manufacturing           student  north_america   
...              ...            ...               ...            ...   
4995  organic_search      education          employed           asia   
4996             NaN     technology          employed  north_america   
4997  organic_search        finance          employed           asia   
4998        referral     technology          employed         africa   
4999        referral     technology          employed         europe   

      annual_income  number_of_courses_viewed  interaction_count  lead_score  \
0           88160.0    

In [43]:
df.dtypes

lead_source                     str
industry                        str
employment_status               str
location                        str
annual_income               float64
number_of_courses_viewed      int64
interaction_count             int64
lead_score                  float64
converted                     int64
dtype: object

In [44]:
# Check missing values before splitting the data.
df.isnull().sum()

lead_source                 148
industry                    240
employment_status           194
location                    208
annual_income               369
number_of_courses_viewed      0
interaction_count             0
lead_score                   35
converted                     0
dtype: int64

In [45]:
# Replace missing values from categorical features with 'NA'
categorical_cols = ['lead_source', 'industry', 'employment_status', 'location']
df[categorical_cols] = df[categorical_cols].fillna('NA')
df.isnull().sum()

lead_source                   0
industry                      0
employment_status             0
location                      0
annual_income               369
number_of_courses_viewed      0
interaction_count             0
lead_score                   35
converted                     0
dtype: int64

In [46]:
# Replace missing values from numerical features with 0.0
numerical_cols = ['annual_income', 'lead_score']
df[numerical_cols] = df[numerical_cols].fillna(0.0)
df.isnull().sum()

lead_source                 0
industry                    0
employment_status           0
location                    0
annual_income               0
number_of_courses_viewed    0
interaction_count           0
lead_score                  0
converted                   0
dtype: int64

In [47]:
df.columns = df.columns.str.lower().str.replace(' ', '_') # Standardizes column headers.

categorical_columns = list(df.dtypes[df.dtypes == 'object'].index)

for c in categorical_columns:
    df[c] = df[c].str.lower().str.replace(' ', '_') # Standardizes object column data but not object data types in this df.

In [48]:
# This safely targets both older 'object' text and newer 'string' text
categorical_columns = list(df.select_dtypes(include=['object', 'string']).columns)

for c in categorical_columns:
    df[c] = df[c].str.lower().str.replace(' ', '_') # Standardizes object column data.

In [49]:
df.head().T  # Transposes the first 5 rows of the DataFrame for easier vertical reading

,0,1,2,3,4
lead_source,organic_search,social_media,referral,paid_ads,referral
industry,technology,technology,retail,manufacturing,manufacturing
employment_status,employed,employed,employed,student,student
location,europe,south_america,europe,na,north_america
annual_income,88160.0,72688.0,44697.0,0.0,24062.0
number_of_courses_viewed,3,3,3,3,4
interaction_count,4,7,4,6,5
lead_score,0.64,0.72,0.58,0.57,0.62
converted,1,1,1,0,1


In [50]:
# Check unique values
for col in df.columns:
    print(col)
    print(df[col].unique()[:5])
    print(df[col].nunique())
    print()

lead_source
<StringArray>
['organic_search', 'social_media', 'referral', 'paid_ads', 'na']
Length: 5, dtype: str
6

industry
<StringArray>
['technology', 'retail', 'manufacturing', 'education', 'healthcare']
Length: 5, dtype: str
7

employment_status
<StringArray>
['employed', 'student', 'na', 'unemployed', 'self_employed']
Length: 5, dtype: str
5

location
<StringArray>
['europe', 'south_america', 'na', 'north_america', 'africa']
Length: 5, dtype: str
6

annual_income
[88160. 72688. 44697.     0. 24062.]
4314

number_of_courses_viewed
[3 4 2 1 0]
7

interaction_count
[4 7 6 5 9]
14

lead_score
[0.64 0.72 0.58 0.57 0.62]
100

converted
[1 0]
2



## Question 1
- What is the most frequent observation (mode) for the column industry?
    - `NA`
    - `technology`
    - `healthcare`
    - `retail`

In [51]:
df["industry"].value_counts().head()

industry
technology    1173
retail         925
healthcare     840
finance        720
education      639
Name: count, dtype: int64

## Question 2
- Create the correlation matrix for the numerical features of your dataset. In a correlation matrix, you compute the correlation coefficient between every pair of features.
What are the two features that have the biggest correlation?
    - `interaction_count` and `lead_score`
    - `number_of_courses_viewed` and `lead_score`
    - `number_of_courses_viewed` and `interaction_count`
    - `annual_income` and `interaction_count`
Only consider the pairs above when answering this question.

In [52]:
numerical_columns = ['annual_income', 'interaction_count', 'number_of_courses_viewed', 'lead_score']
correlation_matrix = df[numerical_columns].corr()
correlation_matrix

,annual_income,interaction_count,number_of_courses_viewed,lead_score
annual_income,1.000000,0.122842,0.161300,0.229496
interaction_count,0.122842,1.000000,0.721609,0.915746
number_of_courses_viewed,0.161300,0.721609,1.000000,0.757204
lead_score,0.229496,0.915746,0.757204,1.000000


## Split the data

In [53]:
from sklearn.model_selection import train_test_split

In [54]:
# Splits the original dataset, setting aside 20% for testing and keeping the remaining 80% as the full training set.
# Setting random_state=42 ensures the random shuffle is reproducible if the notebook is run again (i.e. seeding).
df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=42)

# Splits the 80% full training set into the final training set and a validation set. 
# The test_size=0.25 parameter is used because 25% of the 80% full training data equals exactly 20% of the original dataset.
# This two-step process results in the desired 60/20/20 split: 60% for training, 20% for validation, and 20% for testing.
df_train, df_val = train_test_split(df_full_train, test_size=0.25, random_state=42)

In [55]:
len(df_train), len(df_val), len(df_test) # Confirming dataset sizes

(3000, 1000, 1000)

In [56]:
df_train = df_train.reset_index(drop=True) # Gives a new index and drop true avoids the old index being added as a new column.
df_val = df_val.reset_index(drop=True)
df_test = df_test.reset_index(drop=True)
df_train.head()

,lead_source,industry,employment_status,location,annual_income,number_of_courses_viewed,interaction_count,lead_score,converted
0,paid_ads,finance,employed,asia,65980.0,2,3,0.37,1
1,events,finance,employed,south_america,71662.0,0,0,0.09,0
2,referral,na,employed,asia,40220.0,2,5,0.55,1
3,organic_search,finance,employed,north_america,88352.0,2,2,0.45,0
4,events,technology,employed,north_america,89316.0,1,5,0.00,1


In [57]:
y_train = df_train.converted.values
y_val = df_val.converted.values
y_test = df_test.converted.values

del df_train['converted']
del df_val['converted']
del df_test['converted']
y_train

array([1, 0, 1, ..., 1, 0, 0], shape=(3000,))

In [58]:
df_train.head()

,lead_source,industry,employment_status,location,annual_income,number_of_courses_viewed,interaction_count,lead_score
0,paid_ads,finance,employed,asia,65980.0,2,3,0.37
1,events,finance,employed,south_america,71662.0,0,0,0.09
2,referral,na,employed,asia,40220.0,2,5,0.55
3,organic_search,finance,employed,north_america,88352.0,2,2,0.45
4,events,technology,employed,north_america,89316.0,1,5,0.00


## EDA

In [59]:
# The full training set (combining both training and validation sets) is used for exploratory data analysis (EDA) because it will be needed to compute feature importance later[cite: 1].
# We reset the index to provide a clean sequential sequence, using drop=True to completely discard the old, shuffled index instead of keeping it as a new column[cite: 1, 2].
df_full_train = df_full_train.reset_index(drop=True)

In [60]:
df_full_train.converted.value_counts(normalize=True)

converted
1    0.57075
0    0.42925
Name: proportion, dtype: float64

In [61]:
df_full_train.converted.mean()

np.float64(0.57075)

In [62]:
# Checking the number of unique values for each categorical variable is important
# because it dictates how many new binary columns will be generated during the one-hot encoding process later.
df_full_train[categorical_cols].nunique()

lead_source          6
industry             7
employment_status    5
location             6
dtype: int64

## Question 3
- Calculate the mutual information score between converted and other categorical variables in the dataset. Use the training set only.
- Round the scores to 2 decimals using `round(score, 2)`.
Which of these variables has the biggest mutual information score?
    - `industry`
    - `location`
    - `lead_source`
    - `employment_status`

In [63]:
from sklearn.metrics import mutual_info_score

In [68]:
def calculate_mi(series):
    return mutual_info_score(series, y_train)

In [69]:
mi_scores = df_train[categorical_cols].apply(calculate_mi).round(2)
mi_scores = mi_scores.sort_values(ascending=False)
print(mi_scores)

lead_source          0.03
employment_status    0.02
industry             0.00
location             0.00
dtype: float64


## Question 4
- Now let's train a logistic regression.
- Remember that we have several categorical variables in the dataset. Include them using one-hot encoding.
- Fit the model on the training dataset.
    - To make sure the results are reproducible across different versions of Scikit-Learn, fit the model with these parameters:
    - `model = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)`
- Calculate the accuracy on the validation dataset and round it to 2 decimal digits.
What accuracy did you get?
    - `0.55`
    - `0.65`
    - `0.75`
    - `0.85`

In [71]:
from sklearn.feature_extraction import DictVectorizer
# DictVectorizer is used to do one-hot encoding, where feature categories are turned into multiple columns of 0s and 1s
# DictVectorizer takes a list of dictionaries, one dictionary per row

In [72]:
# One-hot encoding creates mostly 0s, so it defaults to a memory-efficient sparse matrix that only stores 1s.
# Setting sparse=False forces a dense NumPy array instead, which is easier to work with for this dataset.
dv = DictVectorizer(sparse=False)

train_dict = df_train[categorical_cols + numerical_columns].to_dict(orient='records')
X_train = dv.fit_transform(train_dict) # fit learns the list of categories for each column, fit_transform does both in one call

val_dict = df_val[categorical_cols + numerical_columns].to_dict(orient='records')
X_val = dv.transform(val_dict) # transform reuses the previous fit and turns dictionaries into a matrix

In [73]:
X_train.shape

(3000, 28)

In [102]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

In [107]:
model = LogisticRegression(
    solver="liblinear",
    C=1.0,
    max_iter=1000,
    random_state=42
)

model.fit(X_train, y_train)

,"random_state random_state: int, RandomState instance, default=NoneOnly used for `solver` == 'sag', 'saga' or 'liblinear' to shuffle thedata. It has no effect on the other solvers.See :term:`Glossary <random_state>` for details.",42
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver because it works reasonably well for a wide class of problems.- For :term:`multiclass` problems (`n_classes >= 3`), all solvers except 'liblinear' minimize the full multinomial loss, 'liblinear' will raise an error.- 'newton-cholesky' is a good choice for `n_samples` >> `n_features * n_classes`, especially with one-hot encoded categorical features with rare categories. Be aware that the memory usage of this solver has a quadratic dependency on `n_features * n_classes` because it explicitly computes the full Hessian matrix.- For small datasets, 'liblinear' is a good choice, whereas 'sag' and 'saga' are faster for large ones;- 'liblinear' can only handle binary classification by default. To apply a one-versus-rest scheme for the multiclass setting one can wrap it with the :class:`~sklearn.multiclass.OneVsRestClassifier`... warning:: The choice of the algorithm depends on the penalty chosen (`l1_ratio=0` for L2-penalty, `l1_ratio=1` for L1-penalty and `0 < l1_ratio < 1` for Elastic-Net) and on (multinomial) multiclass support: ================= ======================== ====================== solver l1_ratio multinomial multiclass ================= ======================== ====================== 'lbfgs' l1_ratio=0 yes 'liblinear' l1_ratio=1 or l1_ratio=0 no 'newton-cg' l1_ratio=0 yes 'newton-cholesky' l1_ratio=0 yes 'sag' l1_ratio=0 yes 'saga' 0<=l1_ratio<=1 yes ================= ======================== ======================.. note:: 'sag' and 'saga' fast convergence is only guaranteed on features with approximately the same scale. You can preprocess the data with a scaler from :mod:`sklearn.preprocessing`... seealso:: Refer to the :ref:`User Guide <Logistic_regression>` for more information regarding :class:`LogisticRegression` and more specifically the :ref:`Table <logistic_regression_solvers>` summarizing solver/penalty supports... versionadded:: 0.17 Stochastic Average Gradient (SAG) descent solver. Multinomial support in version 0.18... versionadded:: 0.19 SAGA solver... versionchanged:: 0.22 The default solver changed from 'liblinear' to 'lbfgs' in 0.22... versionadded:: 1.2 newton-cholesky solver. Multinomial support in version 1.6.",'liblinear'
,"max_iter max_iter: int, default=100Maximum number of iterations taken for the solvers to converge.",1000
,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add an L2 penalty term and it is the default choice;- `'l1'`: add an L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` and `C` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'`, `l1_ratio` set to any float between 0 and 1 for `penalty='elasticnet'`, and `C=np.inf` for `penalty=None`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l

In [113]:
y_pred = model.predict(X_val)

In [114]:
y_pred_proba = model.predict_proba(X_val)[:, 1]

In [115]:
convertion = (y_pred_proba >= 0.5)

In [116]:
accuracy = (y_val == convertion).mean()
print(round(accuracy, 2))

0.64


In [117]:
accuracy = accuracy_score(y_val, y_pred)
round(accuracy, 2)

0.65

## Question 5
- Let's find the least useful feature using the feature elimination technique.
- Train a model using the same features and parameters as in Q4 (without rounding).
- Now exclude each feature from this set and train a model without it. Record the accuracy for each model.
- For each feature, calculate the difference between the original accuracy and the accuracy without the feature.
Which of following feature has the smallest difference?
    - `lead_source`
    - `number_of_courses_viewed`
    - `interaction_count`
 > Note: The difference doesn't have to be positive.

In [124]:
features_all = categorical_cols + numerical_columns
features_all

['lead_source',
 'industry',
 'employment_status',
 'location',
 'annual_income',
 'interaction_count',
 'number_of_courses_viewed',
 'lead_score']

In [125]:
def calculate_accuracy(features):
    dv = DictVectorizer(sparse=False)

    train_dicts = df_train[features].to_dict(orient="records")
    X_train = dv.fit_transform(train_dicts)

    val_dicts = df_val[features].to_dict(orient="records")
    X_val = dv.transform(val_dicts)
    model = LogisticRegression(
        solver="liblinear",
        C=1.0,
        max_iter=1000,
        random_state=42
    )
    model.fit(X_train, y_train)
    y_pred = model.predict(X_val)
    accuracy = accuracy_score(y_val, y_pred)
    return accuracy

In [129]:
original_accuracy = calculate_accuracy(features_all)
original_accuracy

0.645

In [130]:
eliminated_accuracy_change = {}
for feature in features_all:
    subset = [f for f in features_all if f != feature]
    accuracy = calculate_accuracy(subset)
    difference = original_accuracy - accuracy
    eliminated_accuracy_change[feature] = difference
eliminated_accuracy_change

{'lead_source': 0.0030000000000000027,
 'industry': 0.0,
 'employment_status': 0.0,
 'location': 0.0,
 'annual_income': -0.07899999999999996,
 'interaction_count': 0.04400000000000004,
 'number_of_courses_viewed': 0.0020000000000000018,
 'lead_score': 0.0010000000000000009}

## Question 6
- Now let's train a regularized logistic regression.
- Let's try the following values of the parameter C: [0.000001, 0.00001, 0.0001, 0.001].
- Train models using all the features as in Q4.
- Calculate the accuracy on the validation dataset and round it to 3 decimal digits.
Which of these `C` leads to the best accuracy on the validation set?
    - `0.000001`
    - `0.00001`
    - `0.0001`
    - `0.001`
Note: If there are multiple options, select the smallest C.

In [131]:
def calculate_regularized_accuracy(features, c):
    dv = DictVectorizer(sparse=False)

    train_dicts = df_train[features].to_dict(orient="records")
    X_train = dv.fit_transform(train_dicts)

    val_dicts = df_val[features].to_dict(orient="records")
    X_val = dv.transform(val_dicts)
    model = LogisticRegression(
        solver="liblinear",
        C=c,
        max_iter=1000,
        random_state=42
    )
    model.fit(X_train, y_train)
    y_pred = model.predict(X_val)
    accuracy = accuracy_score(y_val, y_pred)
    return accuracy

In [140]:
regularized_logistic_regression_accuracy = {}
for C in [0.000001, 0.00001, 0.0001, 0.001]:
    regularized_logistic_regression_accuracy[C] = round(calculate_regularized_accuracy(features_all, C),3)
regularized_logistic_regression_accuracy

{1e-06: 0.598, 1e-05: 0.598, 0.0001: 0.613, 0.001: 0.645}

In [141]:
for c, acc in regularized_logistic_regression_accuracy.items():
    print(f"C: {c:.6f}  -->  Accuracy: {acc:.3f}")

C: 0.000001  -->  Accuracy: 0.598
C: 0.000010  -->  Accuracy: 0.598
C: 0.000100  -->  Accuracy: 0.613
C: 0.001000  -->  Accuracy: 0.645
